In [ ]:
#Load
import pandas as pd

df = pd.read_excel(r'C:\Users\22204093\Downloads\Restaurant_orders.xlsx')
df

In [ ]:
#Data Cleaning
df.isna().mean()*100

In [ ]:
df.dtypes

In [ ]:
df.duplicated().sum()

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)
df.columns

In [ ]:
df['discounts_and_offers'] = df['discounts_and_offers'].fillna(0)
df.isna().mean()*100

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['payment_method_n'] = le.fit_transform(df['payment_method'])

In [ ]:
df['year'] = df['order_date_and_time'].dt.year
df['month'] = df['order_date_and_time'].dt.month
df['day'] = df['order_date_and_time'].dt.day
df.head()

In [ ]:
df['total_revenue'] = df['order_value'] - df['delivery_fee'] - df['payment_processing_fee'] - df['refunds/chargebacks']
df.head()

In [ ]:
#Correlation Heatmap
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

In [ ]:
#Distribution of Order Value
df['order_value'].hist(bins=5)
plt.title('Distribution of Order Value')
plt.xlabel('Order Value')
plt.ylabel('Count')
plt.show()

In [ ]:
#Payment Method Distribution
sns.countplot(x='payment_method', data=df)
plt.show()

In [ ]:
#Refunds by Payment Method
df['refund_flag'] = (df['refunds/chargebacks'] > 0).astype(int)
sns.countplot(x='payment_method', hue='refund_flag', data=df)
plt.show()

In [ ]:
#Order Value by Payment Method
sns.boxplot(x='payment_method', y='order_value', data=df)
plt.show()

In [ ]:
#Order Value vs Delivery Fee
sns.scatterplot(x='order_value', y='delivery_fee', data=df)
plt.show()

In [ ]:
#Average Order Value by Payment Method
sns.barplot(x='payment_method', y='order_value', data=df)
plt.show()

In [ ]:
#Revenue by month
df.groupby(['month'])[['total_revenue']].sum().sort_values(['total_revenue'], ascending=False)

In [ ]:
#Average order value by payment method
df.groupby(['payment_method'])[['order_value']].mean().sort_values(['order_value'], ascending=False)

In [ ]:
#Linear Regression - predict total_revenue
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X = df[['order_value', 'delivery_fee', 'payment_method_n', 'payment_processing_fee', 'month', 'day']]
y = df['total_revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
model.score(X_test, y_test)

In [ ]:
#Logistic Regression - predict refund (0/1)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X = df[['order_value', 'delivery_fee', 'payment_method_n', 'payment_processing_fee', 'month', 'day']]
y = df['refund_flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model2 = LogisticRegression(max_iter=10000)
model2.fit(X_train, y_train)
model2.score(X_test, y_test)

In [ ]:
#Predict
model.predict([[500, 30, 2, 25, 6, 15]])